# Agoda Reviews — BERTopic Pipeline

**Stack**: `paraphrase-multilingual-mpnet-base-v2` → Qdrant (Docker) → BERTopic

**Text fields**: `comment` (raw, stored for display) + `processed_comment` (ViTokenizer/spaCy, used for c-TF-IDF)  
**Qdrant**: Docker at `localhost:6333`  collection: `agoda_reviews_envi`

---
## Prerequisites
```bash
# 1. Start Qdrant via Docker
docker run -p 6333:6333 -p 6334:6334 \
    -v $(pwd)/qdrant_storage:/qdrant/storage \
    qdrant/qdrant

# 2. Install dependencies
uv add bertopic sentence-transformers qdrant-client umap-learn hdbscan scikit-learn matplotlib stopwordsiso pyvi spacy
uv run python -m spacy download en_core_web_sm
```

## Section 1 — Imports & Config

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from topic_modeling import (
    Preprocessor,
    ReviewLoader,
    EmbeddingEngine,
    QdrantStore,
    build_bertopic,
    run_pipeline,
    COLLECTION_NAME,
)

# ── Config ────────────────────────────────────────────────────────────────
CSV_PATH    = Path("../data/agoda-reviews-en-vi.csv")
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
NR_TOPICS   = "auto"
LANGUAGE    = "vi"   # "vi" or "en"

# min_cluster_size: keep small for vi (~139 docs), larger for en (~8993 docs)
MIN_CLUSTER_SIZE = 10 if LANGUAGE == "vi" else 50

print(f"Config OK  |  language='{LANGUAGE}'  |  min_cluster_size={MIN_CLUSTER_SIZE}")

Config OK  |  language='vi'  |  min_cluster_size=10


## Section 2 — Preprocess → Embed → Store in Qdrant

`Preprocessor` runs first (ViTokenizer for `vi`, spaCy for `en`) so CountVectorizer
sees `khách_sạn` instead of `khách sạn`.  Both `comment` (raw) and
`processed_comment` are stored in the Qdrant payload.

Run once — safe to re-run, skips if collection already populated.

In [2]:
store = QdrantStore(host=QDRANT_HOST, port=QDRANT_PORT)
store.create_collection(recreate=False)  # wipes existing collection and rebuilds

# 1. Load en/vi reviews
loader = ReviewLoader(CSV_PATH)
df_all = loader.load()

# 2. Preprocess: normalize + ViTokenizer (vi) / spaCy (en)
preprocessor = Preprocessor()
processed_texts = preprocessor.process(df_all)
df_all["processed_comment"] = processed_texts

# 3. Embed the preprocessed text
engine = EmbeddingEngine(batch_size=64)
embeddings_all = engine.encode(processed_texts)

# 4. Upsert — payload contains both comment (raw) and processed_comment
store.upsert(df_all, embeddings_all)
print(f"Stored {len(processed_texts):,} reviews in Qdrant  (collection='{COLLECTION_NAME}').")

[QdrantStore] Connected to Qdrant at localhost:6333  collection='agoda_reviews_envi'
[QdrantStore] Collection 'agoda_reviews_envi' already exists — skipping creation.
[ReviewLoader] Loaded 125,347 reviews with valid comments.
[Preprocessor] Loaded ViTokenizer + spaCy en_core_web_sm.
[Preprocessor] Processing 125,347 rows …


KeyboardInterrupt: 

## Section 3 — Load Embedding Model + Fetch from Qdrant (filtered by `LANGUAGE`)

In [3]:
# Ensure the embedding model is loaded (needed by KeyBERTInspired in Section 4)
if "engine" not in dir():
    engine = EmbeddingEngine(batch_size=64)

# Fetch ALL points from Qdrant, then filter by language with pandas
all_points = []
offset = None

while True:
    results, next_offset = store.client.scroll(
        collection_name=COLLECTION_NAME,
        with_vectors=True,
        with_payload=True,
        limit=1000,
        offset=offset,
    )
    all_points.extend(results)
    if next_offset is None:
        break
    offset = next_offset

df_points      = pd.DataFrame([p.payload for p in all_points])
embeddings_all = np.array([p.vector for p in all_points], dtype=np.float32)

# Filter by language — use processed_comment for BERTopic (c-TF-IDF sees khách_sạn not khách sạn)
mask       = df_points["language"] == LANGUAGE
all_points = [p for p, m in zip(all_points, mask) if m]
docs       = df_points.loc[mask, "processed_comment"].tolist()
embeddings = embeddings_all[mask.values]

print(f"Language   : '{LANGUAGE}'")
print(f"Documents  : {len(docs):,}")
print(f"Embeddings : {embeddings.shape}  dtype={embeddings.dtype}")
print(f"\nSample processed doc:\n  {docs[0][:120]}")

[EmbeddingEngine] Loading model: paraphrase-multilingual-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Language   : 'vi'
Documents  : 39,025
Embeddings : (39025, 768)  dtype=float32

Sample processed doc:
  rất thích ks vì vị_trí gần trung_tâm phòng sạch_sẽ yên_tĩnh đặc_biệt mình thích giường ở ks vì nằm k bị đau lưng tuy_nhi


In [4]:
SAMPLE_SIZE = 10_000
if len(docs) > SAMPLE_SIZE:
    import random
    random.seed(42)
    idx        = random.sample(range(len(docs)), SAMPLE_SIZE)
    docs       = [docs[i] for i in idx]
    embeddings = embeddings[idx]
    print(f"Sampled {SAMPLE_SIZE:,} from {mask.sum():,} '{LANGUAGE}' documents.")

print(f"Language   : '{LANGUAGE}'")
print(f"Documents  : {len(docs):,}")
print(f"Embeddings : {embeddings.shape}  dtype={embeddings.dtype}")
print(f"\nSample processed doc:\n  {docs[0][:120]}")

Sampled 10,000 from 39,025 'vi' documents.
Language   : 'vi'
Documents  : 10,000
Embeddings : (10000, 768)  dtype=float32

Sample processed doc:
  mình đi 6 ng khách_sạn phục_vụ quản_lí vui_tính nhiệt_tình chi moi toi nhân_viên nam lễ_tân hơi chậm nhưng ko sao ns chu


## Section 4 — Fit BERTopic

Pre-fetched `embeddings` passed directly → `embedding_model=None` skips re-encoding.

In [5]:
topic_model = build_bertopic(
    nr_topics=NR_TOPICS,
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_topic_size=MIN_CLUSTER_SIZE,
    embedding_model=engine.model,   # needed by KeyBERTInspired; clustering still uses pre-computed embeddings
)
topics, probs = topic_model.fit_transform(docs, embeddings)

n_topics  = len(set(topics)) - (1 if -1 in topics else 0)
n_outlier = sum(1 for t in topics if t == -1)

print(f"\nLanguage     : '{LANGUAGE}'")
print(f"Topics found : {n_topics}")
print(f"Outliers     : {n_outlier:,} ({n_outlier/len(topics)*100:.1f}%)")

2026-03-25 14:02:47,617 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


[build_bertopic] BERTopic configured.


2026-03-25 14:03:08,695 - BERTopic - Dimensionality - Completed ✓
2026-03-25 14:03:08,696 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-25 14:03:20,554 - BERTopic - Cluster - Completed ✓
2026-03-25 14:03:20,555 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-03-25 14:03:20,806 - BERTopic - Representation - Completed ✓
2026-03-25 14:03:20,807 - BERTopic - Topic reduction - Reducing number of topics
2026-03-25 14:03:20,919 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-25 14:03:56,255 - BERTopic - Representation - Completed ✓
2026-03-25 14:03:56,257 - BERTopic - Topic reduction - Reduced number of topics from 276 to 69



Language     : 'vi'
Topics found : 68
Outliers     : 3,121 (31.2%)


## Section 5 — Inspect Topic Info

In [7]:
topic_info = topic_model.get_topic_info()
print(f"Shape: {topic_info.shape}")
topic_info.head(5)

Shape: (69, 7)


,Topic,Count,Name,Representation,KeyBERT,MMR,Representative_Docs
0,-1,3121,-1_phòng_khách_sạn_nhân_viên_sạch_sẽ,"[phòng, khách_sạn, nhân_viên, sạch_sẽ, gần, ks...","[phòng_ốc, phòng sạch_sẽ, phòng, khách, sạch_s...","[phòng, khách_sạn, sạch_sẽ, đẹp, sáng, ăn, nhi...",[mình đi tới đây để dự đám_cưới ngày đầu tới t...
1,0,4953,0_phòng_nhân_viên_khách_sạn_sạch_sẽ,"[phòng, nhân_viên, khách_sạn, sạch_sẽ, nhiệt_t...","[khách, đặt phòng, khách_sạn sạch_sẽ, khách_sạ...","[phòng, khách_sạn, ăn, sáng, tiện_nghi, khách,...",[khách_sạn mới sạch_sẽ nhân_viên nhiệt_tình th...
2,1,187,1_khach_san_phong_khach san,"[khach, san, phong, khach san, toi, rat, nhan,...","[thuan tien, khong, thuong, nguoi, thuan, thoa...","[khach, san, khach san, toi, nhan, tien, khong...",[phong khach san khong vua y noi that trong ph...
3,2,166,2_tuyệt_vời tốt_tuyệt_vời_tốt tốt_tốt tuyệt_vời,"[tuyệt_vời tốt, tuyệt_vời, tốt tốt, tốt tuyệt_...","[tuyệt_vời tốt, tuyệt tốt, tốt tuyệt_vời, tuyệ...","[tốt tốt, tốt tuyệt_vời, tuyệt_vời tuyệt_vời, ...","[tuyệt_vời, tốt quá tốt, rất tốt tuyệt_vời]"
4,3,125,3_ồn_ngủ_cách_âm_nghe,"[ồn, ngủ, cách_âm, nghe, tiếng, bên, kém, nghe...","[ồn_ào phòng, phòng ồn_ào, tiếng phòng, phòng ...","[nghe tiếng, đêm, ồn_ào, phòng bên, khó ngủ, h...",[mọi thứ đều ổn so với giá tiền nhược_điểm đún...


In [ ]:
# Top keywords for a specific topic (change index as needed)
TOPIC_ID = 0
print(f"Topic {TOPIC_ID} keywords:")
for word, score in topic_model.get_topic(TOPIC_ID):
    print(f"  {word:<25} {score:.4f}")

## Section 6 — (Optional) Reduce Topics

Uncomment and set `TARGET` after reviewing auto results.

In [ ]:
# TARGET = 25
# topic_model.reduce_topics(docs, nr_topics=TARGET)
# topics = topic_model.topics_
# print(f"After reduction: {len(set(topics)) - (1 if -1 in topics else 0)} topics")

## Section 7 — Visualizations

In [8]:
# Topic distance map (intertopic distance)
topic_model.visualize_topics()

In [ ]:
# Top-N words per topic as bar chart (first 12 topics)
topic_model.visualize_barchart(top_n_topics=12, n_words=8)

In [ ]:
# Topic hierarchy
topic_model.visualize_hierarchy()

In [ ]:
# Document-level UMAP scatter coloured by topic (sample 3000 for speed)
import random

n_sample = min(3000, len(docs))
idx = random.sample(range(len(docs)), n_sample)

topic_model.visualize_documents(
    [docs[i] for i in idx],
    embeddings=embeddings[idx],
    hide_document_hover=True,
)

## Section 8 — Probability Heatmap

`calculate_probabilities=True` gives soft assignment — each doc has a score for every topic.

In [ ]:
# Shape: (n_docs, n_topics)
print(f"Probability matrix shape: {probs.shape}")

# Mean probability per topic (rough topic prominence)
topic_prominence = pd.DataFrame({
    "topic": list(range(probs.shape[1])),
    "mean_prob": probs.mean(axis=0),
}).sort_values("mean_prob", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(topic_prominence["topic"].astype(str), topic_prominence["mean_prob"])
ax.set_xlabel("Topic")
ax.set_ylabel("Mean probability")
ax.set_title("Topic Prominence (mean soft-assignment probability)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## Section 9 — Per-Hotel Topic Distribution

In [ ]:
# Build df_result from all_points (language-filtered payload) — perfectly aligned with topics/probs
df_result = pd.DataFrame([p.payload for p in all_points])
df_result["topic"]    = topics
df_result["top_prob"] = probs.max(axis=1) if probs.ndim == 2 else probs

print(f"Language : '{LANGUAGE}'  |  rows: {len(df_result):,}")
df_result.head(3)

In [ ]:
HOTEL_ID = df_result["hotel_id"].iloc[0]   # change to any hotel_id in the dataset

hotel_df     = df_result[df_result["hotel_id"] == HOTEL_ID]
topic_counts = hotel_df["topic"].value_counts().drop(-1, errors="ignore")

print(f"Hotel {HOTEL_ID} [{LANGUAGE}] — {len(hotel_df)} reviews across {len(topic_counts)} topics")

fig, ax = plt.subplots(figsize=(8, 4))
topic_counts.plot(kind="bar", ax=ax)
ax.set_xlabel("Topic")
ax.set_ylabel("Review count")
ax.set_title(f"Topic distribution — Hotel {HOTEL_ID}  [{LANGUAGE}]")
plt.tight_layout()
plt.show()

## Section 10 — Save Model

In [ ]:
SAVE_DIR = Path(f"./bertopic_model_{LANGUAGE}")
topic_model.save(str(SAVE_DIR), serialization="safetensors", save_ctfidf=True)
print(f"Model saved to {SAVE_DIR.resolve()}")

# Load back:
# from bertopic import BERTopic
# topic_model = BERTopic.load(str(SAVE_DIR))